Data Prep

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

df = pd.read_csv("Churn.csv")

# Encode categorical variables
df['Geography'] = LabelEncoder().fit_transform(df['Geography'])  # France/Germany/Spain -> 0/1/2
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])        # Female/Male -> 0/1

# Drop identifiers and non-predictive/redundant fields
df_model = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Card Type'])

Defining two features set

In [2]:
# Realistic model: excludes leakage fields
realistic_features = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 
                       'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 
                       'EstimatedSalary']

# Leaky model: includes Complain and Satisfaction Score
leaky_features = realistic_features + ['Complain', 'Satisfaction Score']

y = df_model['Exited']

Train/test split — same split reused for both models, for fair comparison

In [3]:
X_realistic = df_model[realistic_features]
X_leaky = df_model[leaky_features]

X_train_r, X_test_r, y_train, y_test = train_test_split(
    X_realistic, y, test_size=0.2, random_state=42, stratify=y)

X_train_l, X_test_l, _, _ = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
Train the realistic model (Random Forest, class-weighted)

In [4]:
rf_realistic = RandomForestClassifier(n_estimators=200, class_weight='balanced', 
                                        random_state=42)
rf_realistic.fit(X_train_r, y_train)

pred_r = rf_realistic.predict(X_test_r)
proba_r = rf_realistic.predict_proba(X_test_r)[:, 1]

print("=== REALISTIC MODEL (no leakage) ===")
print(classification_report(y_test, pred_r))
print("ROC-AUC:", roc_auc_score(y_test, proba_r))

=== REALISTIC MODEL (no leakage) ===
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1592
           1       0.60      0.59      0.60       408

    accuracy                           0.84      2000
   macro avg       0.75      0.75      0.75      2000
weighted avg       0.84      0.84      0.84      2000

ROC-AUC: 0.8607890863631884


Train the leaky model (same setup, includes Complain + Satisfaction Score)

In [5]:
rf_leaky = RandomForestClassifier(n_estimators=200, class_weight='balanced', 
                                    random_state=42)
rf_leaky.fit(X_train_l, y_train)

pred_l = rf_leaky.predict(X_test_l)
proba_l = rf_leaky.predict_proba(X_test_l)[:, 1]

print("=== LEAKY MODEL (includes Complain) ===")
print(classification_report(y_test, pred_l))
print("ROC-AUC:", roc_auc_score(y_test, proba_l))

=== LEAKY MODEL (includes Complain) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1592
           1       1.00      1.00      1.00       408

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

ROC-AUC: 0.9991416949945807


check the confusion matrix for the realistic model, so we can talk in real customer counts (how many at-risk customers would actually get missed) rather than just percentages

In [6]:
print(confusion_matrix(y_test, pred_r))

[[1430  162]
 [ 166  242]]


SyntaxError: invalid decimal literal (2699750756.py, line 1)